# PID Tuning & Simulation — FOPDT Temperature Loop
**Process:** $G(s) = K_p \cdot e^{-\theta s} / (\tau s + 1)$

This notebook simulates PID control of a First-Order Plus Dead Time (FOPDT) process and compares five tuning methods.

---

## 1. Imports & Setup


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Inline plots
%matplotlib inline

## 2. PID Controller Class
Features: anti-windup (back-calculation), derivative filtering, bumpless transfer.


In [ ]:
class PIDController:
    """PID controller with anti-windup and derivative filtering."""

    def __init__(self, kp=1.0, ki=0.0, kd=0.0, tau_d=0.1,
                 output_limits=(-np.inf, np.inf), dt=1.0):
        self.kp   = kp
        self.ki   = ki
        self.kd   = kd
        self.tau_d = tau_d
        self.dt   = dt
        self.output_min, self.output_max = output_limits
        self.integral       = 0.0
        self.prev_measurement  = None
        self.prev_derivative    = 0.0

    def update(self, setpoint, measurement):
        error = setpoint - measurement

        # Proportional
        p_term = self.kp * error

        # Integral
        self.integral += error * self.dt
        i_term = self.ki * self.integral

        # Derivative on measurement (avoids derivative kick)
        if self.prev_measurement is None:
            d_term = 0.0
        else:
            d_raw = -(measurement - self.prev_measurement) / self.dt
            alpha = self.dt / (self.tau_d + self.dt)
            d_filtered = alpha * d_raw + (1 - alpha) * self.prev_derivative
            self.prev_derivative = d_filtered
            d_term = self.kd * d_filtered

        self.prev_measurement = measurement

        # Unsaturated output
        output_unsat = p_term + i_term + d_term

        # Clamp
        output = np.clip(output_unsat, self.output_min, self.output_max)

        # Back-calculation anti-windup
        if self.ki != 0:
            saturation_error = output - output_unsat
            self.integral += saturation_error / self.ki

        return output

    def reset(self):
        self.integral       = 0.0
        self.prev_measurement = None
        self.prev_derivative  = 0.0

print("PIDController class defined ✅")

## 3. FOPDT Simulation
Dead time is handled with a ring buffer so the delayed input is accurate regardless of `dt`.


In [ ]:
def simulate_fopdt_pid(Kp, tau, theta, dt, Kc, Ki, Kd, T_max, mv_limits=(-10, 10)):
    """Simulate PID control of FOPDT process."""
    n   = int(T_max / dt)
    T   = np.arange(n) * dt
    y   = np.zeros(n)
    u   = np.zeros(n)
    sp  = np.zeros(n)
    sp[10:] = 1.0  # Step setpoint after 10 samples

    # Dead-time ring buffer
    delay_steps = max(int(round(theta / dt)), 1)
    u_delay = np.zeros(delay_steps)
    didx = 0

    pid = PIDController(kp=Kc, ki=Ki, kd=Kd, dt=dt, output_limits=mv_limits)

    for i in range(n):
        # FOPDT: tau * dy/dt + y = Kp * u_delayed
        u_delayed = u_delay[didx % delay_steps]
        if i > 0:
            dy = (Kp * u_delayed - y[i-1]) / tau * dt
            y[i] = y[i-1] + dy

        # PID
        u[i] = pid.update(sp[i], y[i])

        # Advance ring buffer
        u_delay[didx % delay_steps] = u[i]
        didx += 1

    return T, sp, y, u

print("simulate_fopdt_pid defined ✅")

## 4. Tuning Correlations
Implement **Cohen-Coon**, **IMC**, and **Lambda** tuning methods.


In [ ]:
def cohencron_tuning(Kp, tau, theta):
    """Cohen-Coon tuning for PID on FOPDT."""
    tr = theta / tau
    Kc = (1 / Kp) * (1.35 + tr / 12) / (1 + 0.25 * tr)
    tau_i = tau * (2.5 - 2 * tr) / (1 + 0.4 * tr)
    tau_d = tau * (0.37 * tr) / (1 + 0.2 * tr)
    return Kc, Kc / tau_i, Kc * tau_d


def imc_tuning(Kp, tau, theta, lam):
    """IMC tuning. lam = desired closed-loop time constant."""
    Kc = tau / (Kp * (lam + theta))
    Ki = Kc / tau
    Kd = Kc * theta / 2
    return Kc, Ki, Kd


def lambda_tuning(Kp, tau, theta):
    """Lambda tuning (decoupling performance)."""
    lam = max(0.25 * tau, 2 * theta)
    Kc  = tau / (Kp * (lam + theta))
    Ki  = Kc / tau
    tau_d = theta / 2
    Kd  = Kc * tau_d
    return Kc, Ki, Kd

print("Tuning methods defined ✅")

## 5. Define Process Parameters


In [ ]:
# ── Process parameters ────────────────────────────────────────────
Kp   = 0.2    # Process gain
tau  = 600.0  # Time constant (s)
theta = 30.0  # Dead time (s)
dt   = 10.0   # Sample time (s)
T_MAX = 28800 # Simulation duration (s) — 8 hours

print(f"Process: G(s) = {Kp} · e^(-{theta}s) / ({tau}s + 1)")
print(f"  θ/τ = {theta/tau:.4f}")
print(f"  dt  = {dt} s")
print(f"  T   = {T_MAX/3600:.0f} hours")

## 6. Compute Tunings & Simulate


In [ ]:
# ── Compute gains ─────────────────────────────────────────────────
Kc_cc,    Ki_cc,    Kd_cc    = cohencron_tuning(Kp, tau, theta)
Kc_imc60, Ki_imc60, Kd_imc60 = imc_tuning(Kp, tau, theta, lam=60)
Kc_imc120, Ki_imc120, Kd_imc120 = imc_tuning(Kp, tau, theta, lam=120)
Kc_imc180, Ki_imc180, Kd_imc180 = imc_tuning(Kp, tau, theta, lam=180)
Kc_lam,   Ki_lam,   Kd_lam   = lambda_tuning(Kp, tau, theta)

tunings = [
    (Kc_cc,    Ki_cc,    Kd_cc,    "Cohen-Coon",                "#d62728"),
    (Kc_imc60, Ki_imc60, Kd_imc60, "IMC λ=60 (aggressive)",    "#1f77b4"),
    (Kc_imc120,Ki_imc120,Kd_imc120, "IMC λ=120 (moderate)",     "#2ca02c"),
    (Kc_imc180,Ki_imc180,Kd_imc180, "IMC λ=180 (conservative)","#ff7f0e"),
    (Kc_lam,   Ki_lam,   Kd_lam,   "Lambda (decoupled)",        "#9467bd"),
]

print("\n" + "="*68)
print("TUNED GAINS")
print("="*68)
print(f"{'Method':<32} {'Kc':>8} {'Ki':>8} {'Kd':>10} {'τi (s)':>8} {'τd (s)':>8}")
print("-"*68)
for Kc, Ki, Kd, name, _ in tunings:
    tau_i = Kc / Ki if Ki > 0 else float('inf')
    tau_d = Kd / Kc if Kc > 0 else 0
    print(f"{name:<32} {Kc:>8.3f} {Ki:>8.5f} {Kd:>10.3f} {tau_i:>8.1f} {tau_d:>8.2f}")

## 7. Performance Metrics


In [ ]:
def compute_metrics(T, sp, y):
    """Compute IAE, rise time, overshoot, settling time."""
    # Rise time (10% → 90%)
    risep = np.where((y >= 0.1) & (y < 0.9))[0]
    rise = (T[risep[-1]] - T[risep[0]]) / 60 if len(risep) > 1 else float('nan')

    # Overshoot
    overshoot = max((y.max() - 1.0) * 100, 0)

    # Settling (within 1%)
    within = np.where(np.abs(y - 1.0) <= 0.01)[0]
    settle = T[within[-1]] / 60 if len(within) > 0 else float('nan')

    # IAE
    iae = np.sum(np.abs(sp - y)) * (T[1] - T[0]) / 60

    return rise, overshoot, settle, iae


results = []
print("\n" + "="*68)
print("PERFORMANCE METRICS")
print("="*68)
print(f"{'Method':<32} {'Rise(m)':>8} {'%OS':>7} {'Settle(h)':>10} {'IAE':>8}")
print("-"*68)
for Kc, Ki, Kd, name, color in tunings:
    T, sp, y, u = simulate_fopdt_pid(Kp, tau, theta, dt, Kc, Ki, Kd, T_MAX)
    rise, os, settle, iae = compute_metrics(T, sp, y)
    results.append((name, color, T, sp, y, u))
    print(f"{name:<32} {rise:>8.1f} {os:>6.1f}% {settle:>10.2f} {iae:>8.1f}")

## 8. Plot Step Response


In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(13, 8), sharex=True)

for name, color, T, sp, y, u in results:
    axes[0].plot(T/3600, y,  color=color, label=name, linewidth=1.8)
    axes[1].plot(T/3600, u,  color=color, linestyle='--', linewidth=1, alpha=0.8)

axes[0].axhline(1.0,  color='black', ls=':',  alpha=0.5)
axes[0].axhline(0.99, color='gray',  ls='--', alpha=0.3, label="±1% band")
axes[0].axhline(1.01, color='gray',  ls='--', alpha=0.3)
axes[0].set_ylabel('Process Output (normalized)')
axes[0].set_title(f'PID Step Response — G(s) = {Kp}·e^(-{theta}s)/({tau}s+1)')
axes[0].legend(fontsize=8.5, loc='lower right')
axes[0].grid(alpha=0.3)
axes[0].set_xlim(0, T_MAX/3600)
axes[1].set_ylabel('Controller Output (MV)')
axes[1].set_xlabel('Time (hours)')
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Recommendation
| Method | Best For |
|--------|----------|
| **Cohen-Coon** | Fast disturbance rejection, but aggressive |
| **IMC λ=60** | Very tight setpoint tracking, high MV activity |
| **IMC λ=120** | Good balance — recommended starting point |
| **IMC λ=180** | Slow but smooth — for sensitive processes |
| **Lambda** | Decoupled MV — good when valve wear is a concern |

**For a first test:** start with **IMC λ=120** — conservative but responsive.

Too slow? → increase `Kc` slightly  
Oscillation? → decrease `Kc` by 20-30%  
Steady offset? → increase `Ki` very slightly
